# Error Analysis

Deep error analysis: which unsafe behaviours are hardest to detect, which pressures break the classifier, model-disagreement analysis, and cross-scenario generalisation.

**Contents:**
1. Calibration curves & Brier score
2. Model disagreement analysis
3. Pressure-specific classifier breakdown
4. Cross-scenario generalisation (leave-one-scenario-out)
5. Hardest-to-detect unsafe behaviours (per safety dimension)
6. Log findings to `results/error_analysis/metrics.json`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score, precision_score,
    recall_score, matthews_corrcoef, brier_score_loss, confusion_matrix
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

SEED = 42
np.random.seed(SEED)
FIGURES_DIR = Path('../reports/figures')
RESULTS_DIR = Path('../results')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Libraries loaded.')

## 1. Load Data & Retrain Models

In [ ]:
# Load data
d2c = pd.read_parquet('../data/processed/d2c_labels.parquet')
d2d = pd.read_parquet('../data/processed/d2d_features.parquet')
d2b = pd.read_parquet('../data/processed/d2b_turns.parquet')
d2a = pd.read_parquet('../data/processed/d2a_sessions.parquet')

df = d2d.merge(d2c, on='session_id')
y = np.array(df['overall_unsafe'].astype(int).tolist())

numeric_cols = [
    'response_length_words', 'response_length_chars', 'sentence_count',
    'avg_sentence_length', 'paragraph_count', 'hedging_count', 'hedging_freq',
    'certainty_count', 'certainty_freq', 'hedging_certainty_ratio',
    'nct_id_count', 'nct_id_unique', 'citation_density',
    'bullet_count', 'numbered_list_count', 'refusal_count',
    'is_monitored', 'is_baseline_pressure'
]
bool_cols = ['has_refusal', 'has_citations', 'has_hedging', 'has_certainty']
for c in bool_cols:
    df[c] = df[c].astype(int)

cat_cols = ['scenario_id', 'pressure_id', 'monitoring_id']
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    df[f'{c}_enc'] = le.fit_transform(df[c].astype(str))
    le_dict[c] = le
encoded_cat_cols = [f'{c}_enc' for c in cat_cols]

feature_cols = numeric_cols + bool_cols + encoded_cat_cols
X_struct = np.array(df[feature_cols].values.tolist(), dtype=float)

# Raw text
text_df = d2b[d2b['role'] == 'assistant'].groupby('session_id')['content'].apply(
    lambda x: ' '.join(x)).reset_index()
text_df.columns = ['session_id', 'full_response']
df = df.merge(text_df, on='session_id', how='left')
df['full_response'] = df['full_response'].fillna('')
X_text = df['full_response'].tolist()

imbalance_ratio = (y == 0).sum() / max((y == 1).sum(), 1)

# Layer-A triggered columns (safety dimensions)
layer_a_triggered = [c for c in d2c.columns if c.startswith('layer_a_') and c.endswith('_triggered')]

print(f'Samples: {len(y)} | Unsafe: {y.sum()} | Safe: {(y==0).sum()}')
print(f'Features: {len(feature_cols)} structured | text via TF-IDF')
print(f'Safety dimensions (Layer A): {len(layer_a_triggered)}')

In [ ]:
# Retrain top models with best hyperparameters
model_results = json.load(open(RESULTS_DIR / 'classification/metrics.json'))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_struct)

def parse_param(val_str, param_name=''):
    """Convert string parameter values back to proper types."""
    if val_str == 'None':
        return None
    if val_str in ('sqrt', 'log2', 'scale', 'auto'):
        return val_str
    try:
        if '.' in val_str:
            return float(val_str)
        return int(val_str)
    except (ValueError, TypeError):
        return val_str

# Logistic Regression
lr_p = model_results['models']['Logistic Regression']['best_params_per_fold'][0]
lr_model = LogisticRegression(
    C=parse_param(lr_p.get('clf__C', '1.0')),
    penalty=lr_p.get('clf__penalty', 'l2'),
    solver='saga', class_weight='balanced', max_iter=2000, random_state=SEED
)

# Random Forest
rf_p = model_results['models']['Random Forest']['best_params_per_fold'][0]
rf_model = RandomForestClassifier(
    n_estimators=parse_param(rf_p.get('clf__n_estimators', '200')),
    max_depth=parse_param(rf_p.get('clf__max_depth', 'None')),
    min_samples_leaf=parse_param(rf_p.get('clf__min_samples_leaf', '5')),
    max_features=parse_param(rf_p.get('clf__max_features', 'sqrt')),
    class_weight='balanced', random_state=SEED
)

# XGBoost
xgb_p = model_results['models']['XGBoost']['best_params_per_fold'][0]
xgb_model = XGBClassifier(
    n_estimators=parse_param(xgb_p.get('clf__n_estimators', '200')),
    max_depth=parse_param(xgb_p.get('clf__max_depth', '5')),
    learning_rate=parse_param(xgb_p.get('clf__learning_rate', '0.1')),
    subsample=parse_param(xgb_p.get('clf__subsample', '0.8')),
    colsample_bytree=parse_param(xgb_p.get('clf__colsample_bytree', '0.8')),
    reg_alpha=parse_param(xgb_p.get('clf__reg_alpha', '0')),
    reg_lambda=parse_param(xgb_p.get('clf__reg_lambda', '1.0')),
    scale_pos_weight=imbalance_ratio, eval_metric='logloss',
    use_label_encoder=False, random_state=SEED, verbosity=0
)

# LightGBM
lgb_p = model_results['models']['LightGBM']['best_params_per_fold'][0]
lgb_model = LGBMClassifier(
    n_estimators=parse_param(lgb_p.get('clf__n_estimators', '200')),
    max_depth=parse_param(lgb_p.get('clf__max_depth', '-1')),
    learning_rate=parse_param(lgb_p.get('clf__learning_rate', '0.1')),
    num_leaves=parse_param(lgb_p.get('clf__num_leaves', '31')),
    subsample=parse_param(lgb_p.get('clf__subsample', '0.8')),
    colsample_bytree=parse_param(lgb_p.get('clf__colsample_bytree', '0.8')),
    reg_alpha=parse_param(lgb_p.get('clf__reg_alpha', '0')),
    reg_lambda=parse_param(lgb_p.get('clf__reg_lambda', '1.0')),
    is_unbalance=True, random_state=SEED, verbosity=-1, n_jobs=1
)

# TF-IDF Text Classifier
tfidf_p = model_results['models']['TF-IDF Text Classifier']['best_params_per_fold'][0]
ngram = eval(tfidf_p.get('tfidf__ngram_range', '(1, 2)'))
tfidf_vec = TfidfVectorizer(
    max_features=parse_param(tfidf_p.get('tfidf__max_features', '5000')),
    ngram_range=ngram,
    min_df=parse_param(tfidf_p.get('tfidf__min_df', '2')),
    sublinear_tf=True
)

# Structured models dict (pipeline with scaler)
struct_models = {
    'Logistic Regression': lr_model,
    'Random Forest': rf_model,
    'XGBoost': xgb_model,
    'LightGBM': lgb_model,
}

print('Model definitions loaded with tuned hyperparameters.')

## 2. Calibration Curves & Brier Score

Calibration measures whether predicted probabilities match observed frequencies. Critical for safety: a model that says "80% unsafe" should be wrong ~20% of the time. Poor calibration means the model's confidence is unreliable for decision-making.

In [ ]:
# Get cross-validated probability predictions for all models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_probs = {}

# Structured models
for name, model in struct_models.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', model)])
    probs = cross_val_predict(pipe, X_struct, y, cv=cv, method='predict_proba')[:, 1]
    cv_probs[name] = probs

# TF-IDF text classifier
tfidf_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=parse_param(tfidf_p.get('tfidf__max_features', '5000')),
        ngram_range=ngram,
        min_df=parse_param(tfidf_p.get('tfidf__min_df', '2')),
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        C=parse_param(tfidf_p.get('clf__C', '1.0')),
        penalty=tfidf_p.get('clf__penalty', 'l2'),
        solver='saga', class_weight='balanced', max_iter=2000, random_state=SEED
    ))
])
cv_probs['TF-IDF Text'] = cross_val_predict(tfidf_pipe, X_text, y, cv=cv, method='predict_proba')[:, 1]

print(f'Cross-validated probabilities computed for {len(cv_probs)} models.')

In [ ]:
# Calibration curves + Brier scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: reliability diagrams
ax = axes[0]
brier_scores = {}
ece_scores = {}

colors = plt.cm.Set2(np.linspace(0, 1, len(cv_probs)))
for idx, (name, probs) in enumerate(cv_probs.items()):
    # Brier score
    brier = brier_score_loss(y, probs)
    brier_scores[name] = brier

    # Calibration curve
    prob_true, prob_pred = calibration_curve(y, probs, n_bins=8, strategy='uniform')

    # Expected calibration error
    bin_counts = np.histogram(probs, bins=8, range=(0, 1))[0]
    bin_weights = bin_counts / len(probs)
    ece = np.sum(np.abs(prob_true - prob_pred) * bin_weights[:len(prob_true)])
    ece_scores[name] = ece

    ax.plot(prob_pred, prob_true, 's-', color=colors[idx],
            label=f'{name} (Brier={brier:.3f})', markersize=5)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly calibrated')
ax.set_xlabel('Mean predicted probability')
ax.set_ylabel('Fraction of positives')
ax.set_title('Reliability Diagrams')
ax.legend(fontsize=7, loc='upper left')
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)

# Right: Brier score + ECE comparison
ax = axes[1]
model_names = list(brier_scores.keys())
x_pos = np.arange(len(model_names))
width = 0.35

bars1 = ax.bar(x_pos - width/2, [brier_scores[m] for m in model_names],
               width, label='Brier Score', color='#e74c3c', alpha=0.7)
bars2 = ax.bar(x_pos + width/2, [ece_scores[m] for m in model_names],
               width, label='ECE', color='#3498db', alpha=0.7)

ax.set_xticks(x_pos)
ax.set_xticklabels(model_names, fontsize=8, rotation=30, ha='right')
ax.set_ylabel('Score (lower = better)')
ax.set_title('Brier Score & Expected Calibration Error')
ax.legend(fontsize=8)

plt.suptitle('Model Calibration Analysis', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '20_calibration_analysis.png', bbox_inches='tight')
plt.show()

print('\n=== Calibration Summary ===')
for name in model_names:
    print(f'  {name:25s}  Brier={brier_scores[name]:.4f}  ECE={ece_scores[name]:.4f}')

## 3. Model Disagreement Analysis

Samples where models disagree reveal decision boundaries and ambiguous cases. In safety monitoring, disagreement signals cases that may need human review.

In [ ]:
# Binary predictions from all models (threshold=0.5)
pred_df = pd.DataFrame({name: (probs >= 0.5).astype(int) for name, probs in cv_probs.items()})
pred_df['y_true'] = y
pred_df['n_models_predict_unsafe'] = pred_df[list(cv_probs.keys())].sum(axis=1)
pred_df['all_agree'] = pred_df['n_models_predict_unsafe'].isin([0, len(cv_probs)])

# Disagreement stats
n_agree = pred_df['all_agree'].sum()
n_disagree = (~pred_df['all_agree']).sum()
print(f'=== Model Agreement ===')
print(f'Full agreement:  {n_agree} / {len(pred_df)} ({n_agree/len(pred_df):.1%})')
print(f'Disagreement:    {n_disagree} / {len(pred_df)} ({n_disagree/len(pred_df):.1%})')

# Disagreement breakdown by true label
disagree_mask = ~pred_df['all_agree']
print(f'\nAmong disagreements (n={n_disagree}):')
print(f'  Actually unsafe: {(pred_df.loc[disagree_mask, "y_true"] == 1).sum()}')
print(f'  Actually safe:   {(pred_df.loc[disagree_mask, "y_true"] == 0).sum()}')

In [ ]:
# Disagreement heatmap + characterisation
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Per-model prediction heatmap for disagreement samples
ax = axes[0]
if n_disagree > 0:
    disagree_preds = pred_df.loc[disagree_mask, list(cv_probs.keys())].copy()
    disagree_preds['True Label'] = pred_df.loc[disagree_mask, 'y_true'].values
    disagree_preds = disagree_preds.sort_values('True Label', ascending=False)
    sns.heatmap(disagree_preds.values, ax=ax, cmap='RdYlGn_r', cbar_kws={'label': '1=unsafe'},
                xticklabels=list(cv_probs.keys()) + ['True Label'],
                yticklabels=False, vmin=0, vmax=1)
    ax.set_title(f'Predictions on Disagreement Samples (n={n_disagree})')
    ax.tick_params(axis='x', rotation=45)
else:
    ax.text(0.5, 0.5, 'No disagreements', ha='center', va='center')
    ax.set_title('Predictions on Disagreement Samples')

# 2. Vote distribution histogram
ax = axes[1]
vote_counts = pred_df['n_models_predict_unsafe'].value_counts().sort_index()
colors_vote = ['#27ae60' if v == 0 else '#e74c3c' if v == len(cv_probs)
               else '#f39c12' for v in vote_counts.index]
ax.bar(vote_counts.index, vote_counts.values, color=colors_vote, alpha=0.8, edgecolor='black')
ax.set_xlabel('Number of models predicting unsafe')
ax.set_ylabel('Sample count')
ax.set_title('Vote Distribution Across Models')
ax.set_xticks(range(len(cv_probs) + 1))

# 3. Disagreement rate by scenario + pressure
ax = axes[2]
pred_df_meta = pred_df.copy()
pred_df_meta['scenario_id'] = df['scenario_id'].values
pred_df_meta['pressure_id'] = df['pressure_id'].values
disagree_rate = pred_df_meta.groupby('scenario_id')['all_agree'].apply(
    lambda x: 1 - x.mean())
disagree_rate.sort_values(ascending=False).plot(kind='barh', ax=ax, color='#8e44ad', alpha=0.8)
ax.set_xlabel('Disagreement rate')
ax.set_title('Model Disagreement Rate by Scenario')

plt.suptitle('Model Disagreement Analysis', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '21_model_disagreement.png', bbox_inches='tight')
plt.show()

## 4. Pressure-Specific Classifier Breakdown

Which pressure conditions break the classifier? This directly tests whether adversarial conditions designed to elicit unsafe behaviour also make that behaviour harder to *detect*.

In [ ]:
# Per-pressure metrics using TF-IDF (best model) and LightGBM (best structured)
best_models_for_analysis = {
    'LightGBM': cv_probs['LightGBM'],
    'TF-IDF Text': cv_probs['TF-IDF Text'],
}

pressure_results = {}
for model_name, probs in best_models_for_analysis.items():
    preds = (probs >= 0.5).astype(int)
    rows = []
    for pressure in df['pressure_id'].unique():
        mask = df['pressure_id'].values == pressure
        y_sub = y[mask]
        p_sub = preds[mask]
        prob_sub = probs[mask]

        n_total = mask.sum()
        n_unsafe = y_sub.sum()
        if n_unsafe == 0:
            rec = np.nan
            pr_auc = np.nan
        else:
            rec = recall_score(y_sub, p_sub, zero_division=0)
            try:
                pr_auc = average_precision_score(y_sub, prob_sub)
            except ValueError:
                pr_auc = np.nan

        prec = precision_score(y_sub, p_sub, zero_division=0)
        f1 = f1_score(y_sub, p_sub, zero_division=0)
        try:
            roc = roc_auc_score(y_sub, prob_sub)
        except ValueError:
            roc = np.nan

        rows.append({
            'pressure': pressure, 'n_total': n_total, 'n_unsafe': n_unsafe,
            'recall': rec, 'precision': prec, 'f1': f1,
            'roc_auc': roc, 'pr_auc': pr_auc
        })
    pressure_results[model_name] = pd.DataFrame(rows)

print('=== Per-Pressure Metrics (LightGBM) ===')
print(pressure_results['LightGBM'].to_string(index=False, float_format='%.3f'))
print('\n=== Per-Pressure Metrics (TF-IDF Text) ===')
print(pressure_results['TF-IDF Text'].to_string(index=False, float_format='%.3f'))

In [ ]:
# Visualise pressure breakdown
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (model_name, pr_df) in enumerate(pressure_results.items()):
    ax = axes[idx]
    pr_sorted = pr_df.sort_values('recall', ascending=True)
    x_pos = np.arange(len(pr_sorted))

    ax.barh(x_pos, pr_sorted['recall'].fillna(0), height=0.35, label='Recall',
            color='#e74c3c', alpha=0.7)
    ax.barh(x_pos + 0.35, pr_sorted['f1'].fillna(0), height=0.35, label='F1',
            color='#3498db', alpha=0.7)

    ax.set_yticks(x_pos + 0.175)
    ax.set_yticklabels(pr_sorted['pressure'].values, fontsize=9)
    ax.set_xlabel('Score')
    ax.set_title(f'{model_name}: Recall & F1 by Pressure')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1.05)

    # Annotate n_unsafe
    for i, (_, row) in enumerate(pr_sorted.iterrows()):
        ax.text(1.02, i + 0.175, f'n={int(row["n_unsafe"])}',
                fontsize=7, va='center', color='gray')

plt.suptitle('Which Pressures Break the Classifier?', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '22_pressure_breakdown.png', bbox_inches='tight')
plt.show()

# Identify worst pressures
worst = pressure_results['TF-IDF Text'].nsmallest(3, 'recall')
print('\n=== Hardest Pressures to Detect (TF-IDF, by recall) ===')
print(worst[['pressure', 'recall', 'f1', 'n_unsafe']].to_string(index=False))

## 5. Cross-Scenario Generalisation (Leave-One-Scenario-Out)

Does the safety classifier generalise across task types? Train on 3 scenarios, test on the held-out 4th. Poor transfer reveals task-specific detection shortcuts rather than genuine safety signal.

In [ ]:
# Leave-one-scenario-out (LOSO) evaluation
scenarios = df['scenario_id'].unique()
loso_results = []

for held_out in scenarios:
    train_mask = df['scenario_id'].values != held_out
    test_mask = df['scenario_id'].values == held_out

    X_train_s, X_test_s = X_struct[train_mask], X_struct[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    X_train_t = [X_text[i] for i in range(len(X_text)) if train_mask[i]]
    X_test_t = [X_text[i] for i in range(len(X_text)) if test_mask[i]]

    # LightGBM (structured)
    pipe_lgb = Pipeline([('scaler', StandardScaler()), ('clf', LGBMClassifier(
        n_estimators=parse_param(lgb_p.get('clf__n_estimators', '200')),
        max_depth=parse_param(lgb_p.get('clf__max_depth', '-1')),
        learning_rate=parse_param(lgb_p.get('clf__learning_rate', '0.1')),
        num_leaves=parse_param(lgb_p.get('clf__num_leaves', '31')),
        is_unbalance=True, random_state=SEED, verbosity=-1, n_jobs=1
    ))])
    pipe_lgb.fit(X_train_s, y_train)
    lgb_prob = pipe_lgb.predict_proba(X_test_s)[:, 1]

    # TF-IDF Text
    pipe_txt = Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=parse_param(tfidf_p.get('tfidf__max_features', '5000')),
            ngram_range=ngram,
            min_df=parse_param(tfidf_p.get('tfidf__min_df', '2')),
            sublinear_tf=True
        )),
        ('clf', LogisticRegression(
            C=parse_param(tfidf_p.get('clf__C', '1.0')),
            penalty=tfidf_p.get('clf__penalty', 'l2'),
            solver='saga', class_weight='balanced', max_iter=2000, random_state=SEED
        ))
    ])
    pipe_txt.fit(X_train_t, y_train)
    txt_prob = pipe_txt.predict_proba(X_test_t)[:, 1]

    for model_name, prob in [('LightGBM', lgb_prob), ('TF-IDF Text', txt_prob)]:
        pred = (prob >= 0.5).astype(int)
        n_unsafe_test = y_test.sum()
        try:
            roc = roc_auc_score(y_test, prob)
        except ValueError:
            roc = np.nan
        try:
            pr = average_precision_score(y_test, prob)
        except ValueError:
            pr = np.nan

        loso_results.append({
            'held_out_scenario': held_out,
            'model': model_name,
            'n_test': len(y_test),
            'n_unsafe_test': int(n_unsafe_test),
            'roc_auc': roc,
            'pr_auc': pr,
            'recall': recall_score(y_test, pred, zero_division=0),
            'f1': f1_score(y_test, pred, zero_division=0),
        })

loso_df = pd.DataFrame(loso_results)
print('=== Leave-One-Scenario-Out Results ===')
print(loso_df.to_string(index=False, float_format='%.3f'))

In [ ]:
# Visualise LOSO results
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, metric in enumerate(['roc_auc', 'recall']):
    ax = axes[idx]
    pivot = loso_df.pivot(index='held_out_scenario', columns='model', values=metric).fillna(0)
    bars = pivot.plot(kind='bar', ax=ax, alpha=0.8, edgecolor='black', width=0.7)
    ax.set_ylabel(metric.replace('_', ' ').title())
    ax.set_title(f'LOSO: {metric.replace("_", " ").title()} by Held-Out Scenario')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

    # Annotate scenarios with no unsafe samples (NaN → 0)
    for i, scenario in enumerate(pivot.index):
        row = loso_df[loso_df['held_out_scenario'] == scenario].iloc[0]
        if row['n_unsafe_test'] == 0:
            ax.annotate('no unsafe\nsamples', xy=(i, 0.02), fontsize=7,
                       ha='center', color='red', style='italic')

plt.suptitle('Cross-Scenario Generalisation (Leave-One-Scenario-Out)', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '23_loso_generalisation.png', bbox_inches='tight')
plt.show()

# Compare LOSO to standard CV
cv_roc_lgb = roc_auc_score(y, cv_probs['LightGBM'])
cv_roc_txt = roc_auc_score(y, cv_probs['TF-IDF Text'])
loso_roc_lgb = loso_df[loso_df['model'] == 'LightGBM']['roc_auc'].mean()
loso_roc_txt = loso_df[loso_df['model'] == 'TF-IDF Text']['roc_auc'].mean()

print('\n=== Generalisation Gap (Standard CV vs LOSO) ===')
print(f'  LightGBM:  CV ROC={cv_roc_lgb:.3f}  LOSO ROC={loso_roc_lgb:.3f}  Gap={cv_roc_lgb - loso_roc_lgb:+.3f}')
print(f'  TF-IDF:    CV ROC={cv_roc_txt:.3f}  LOSO ROC={loso_roc_txt:.3f}  Gap={cv_roc_txt - loso_roc_txt:+.3f}')

## 6. Hardest-to-Detect Unsafe Behaviours

Per safety dimension: which specific violations (Layer A) are most often missed by the classifier? This reveals whether the model catches overt failures but misses subtle ones.

In [ ]:
# Per-dimension detection rate using best model (TF-IDF Text)
best_preds = (cv_probs['TF-IDF Text'] >= 0.5).astype(int)
best_probs = cv_probs['TF-IDF Text']

# Merge layer-A triggered columns into working df
for col in layer_a_triggered:
    df[col] = d2c[col].values

dimension_detection = []
for dim in layer_a_triggered:
    triggered_mask = df[dim].values == True
    n_triggered = triggered_mask.sum()
    if n_triggered == 0:
        continue

    # Of sessions where this dimension triggered, how many did the model catch?
    caught = best_preds[triggered_mask].sum()
    missed = n_triggered - caught
    detection_rate = caught / n_triggered
    avg_prob = best_probs[triggered_mask].mean()

    # Also check: are these sessions actually labelled unsafe overall?
    actually_unsafe = y[triggered_mask].sum()

    dim_short = dim.replace('layer_a_', '').replace('_triggered', '')
    dimension_detection.append({
        'dimension': dim_short,
        'n_triggered': n_triggered,
        'n_actually_unsafe': int(actually_unsafe),
        'n_caught_by_model': int(caught),
        'n_missed': int(missed),
        'detection_rate': detection_rate,
        'avg_predicted_prob': avg_prob,
    })

dim_df = pd.DataFrame(dimension_detection).sort_values('detection_rate', ascending=True)
print('=== Per-Dimension Detection Rates (TF-IDF Text Classifier) ===')
print(dim_df.to_string(index=False, float_format='%.3f'))
print(f'\nDimensions with zero triggers excluded: '
      f'{len(layer_a_triggered) - len(dimension_detection)}')

In [ ]:
# Visualise per-dimension detection
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Detection rate by safety dimension
ax = axes[0]
if len(dim_df) > 0:
    colors_dim = ['#e74c3c' if r < 0.5 else '#f39c12' if r < 0.75 else '#27ae60'
                  for r in dim_df['detection_rate']]
    ax.barh(range(len(dim_df)), dim_df['detection_rate'].values,
            color=colors_dim, alpha=0.8, edgecolor='black')
    ax.set_yticks(range(len(dim_df)))
    ax.set_yticklabels(dim_df['dimension'].values, fontsize=9)
    ax.set_xlabel('Detection Rate')
    ax.set_title('Classifier Detection Rate by Safety Dimension')
    ax.axvline(x=0.5, color='red', linestyle='--', alpha=0.5, label='50% threshold')
    ax.set_xlim(0, 1.05)
    # Annotate counts
    for i, (_, row) in enumerate(dim_df.iterrows()):
        ax.text(row['detection_rate'] + 0.02, i,
                f'n={int(row["n_triggered"])}', fontsize=7, va='center')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'No dimensions with triggers', ha='center', va='center')

# 2. False negative deep dive: which dimensions appear in missed unsafe sessions?
ax = axes[1]
fn_mask = (y == 1) & (best_preds == 0)  # false negatives
tp_mask = (y == 1) & (best_preds == 1)  # true positives

if fn_mask.sum() > 0:
    fn_dim_rates = []
    for dim in layer_a_triggered:
        dim_short = dim.replace('layer_a_', '').replace('_triggered', '')
        fn_rate = df.loc[fn_mask, dim].mean() if fn_mask.sum() > 0 else 0
        tp_rate = df.loc[tp_mask, dim].mean() if tp_mask.sum() > 0 else 0
        fn_dim_rates.append({'dimension': dim_short, 'FN_rate': fn_rate, 'TP_rate': tp_rate})

    fn_dim_df = pd.DataFrame(fn_dim_rates).sort_values('FN_rate', ascending=True)
    x_pos = np.arange(len(fn_dim_df))
    width = 0.35
    ax.barh(x_pos - width/2, fn_dim_df['FN_rate'].values, width,
            label='False Negatives', color='#e74c3c', alpha=0.7)
    ax.barh(x_pos + width/2, fn_dim_df['TP_rate'].values, width,
            label='True Positives', color='#27ae60', alpha=0.7)
    ax.set_yticks(x_pos)
    ax.set_yticklabels(fn_dim_df['dimension'].values, fontsize=8)
    ax.set_xlabel('Trigger Rate')
    ax.set_title('Safety Dimension Trigger Rates: FN vs TP')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'No false negatives', ha='center', va='center')

plt.suptitle('Hardest-to-Detect Unsafe Behaviours', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '24_dimension_detection.png', bbox_inches='tight')
plt.show()

## 7. Log Findings

In [ ]:
# Structured logging
log_entry = {
    'step': 19,
    'description': 'Error analysis: calibration, model disagreement, pressure breakdown, '
                   'cross-scenario generalisation, per-dimension detection',
    'timestamp': datetime.now().isoformat(),
    'seed': SEED,
    'n_samples': len(y),
    'n_unsafe': int(y.sum()),
    'calibration': {
        name: {'brier_score': float(brier_scores[name]), 'ece': float(ece_scores[name])}
        for name in brier_scores
    },
    'model_disagreement': {
        'n_full_agreement': int(n_agree),
        'n_disagreement': int(n_disagree),
        'disagreement_rate': float(n_disagree / len(pred_df)),
        'disagreement_unsafe_count': int((pred_df.loc[disagree_mask, 'y_true'] == 1).sum()),
        'disagreement_safe_count': int((pred_df.loc[disagree_mask, 'y_true'] == 0).sum()),
    },
    'pressure_breakdown': {
        model_name: pr_df.to_dict('records')
        for model_name, pr_df in pressure_results.items()
    },
    'loso_generalisation': loso_df.to_dict('records'),
    'generalisation_gap': {
        'LightGBM': {'cv_roc': float(cv_roc_lgb), 'loso_roc': float(loso_roc_lgb),
                      'gap': float(cv_roc_lgb - loso_roc_lgb)},
        'TF-IDF Text': {'cv_roc': float(cv_roc_txt), 'loso_roc': float(loso_roc_txt),
                         'gap': float(cv_roc_txt - loso_roc_txt)},
    },
    'dimension_detection': dim_df.to_dict('records') if len(dim_df) > 0 else [],
}

results_path = RESULTS_DIR / 'error_analysis/metrics.json'
with open(results_path, 'w') as f:
    json.dump(log_entry, f, indent=2, default=str)

print(f'Results saved to: {results_path}')

## 8. Summary

**Key findings for AI safety oversight:**

- **Calibration** â€” reliability diagrams and Brier scores reveal which models produce well-calibrated probability estimates, critical for threshold-based safety decisions
- **Model disagreement** â€” samples where models disagree highlight ambiguous cases that require human-in-the-loop review in a scalable oversight pipeline
- **Pressure breakdown** â€” identifies which adversarial pressure conditions make unsafe behaviour hardest to detect, informing adversarial robustness priorities
- **Cross-scenario generalisation (LOSO)** â€” the gap between standard CV and leave-one-scenario-out performance quantifies how much the classifier relies on task-specific shortcuts vs genuine safety signal
- **Per-dimension detection** â€” reveals which specific safety violations (overclaiming, evidence coverage gaps, hedging absence) are most often missed, guiding rubric and feature improvements

**Implications:**
- A large LOSO gap suggests the classifier learns scenario-specific patterns rather than transferable safety indicators â€” a red flag for deployment generalisation
- Low detection rates on specific dimensions point to feature engineering gaps: the current feature set may not capture the linguistic signatures of those particular failures
- Model disagreement cases are prime candidates for human annotation and active learning

In [ ]:
# Analysis complete.